### 📊 Interactive Dashboard using Streamlit & Power BI

This project presents an interactive data dashboard built using both Streamlit and Power BI, combining the flexibility of Python-based dashboard with the powerful visualization capabilities of business intelligence tools.

In [ ]:
# app.py - Main Streamlit Dashboard
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import folium
from streamlit_folium import folium_static
import geopandas as gpd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
import joblib
import warnings
warnings.filterwarnings('ignore')

# Page configuration
st.set_page_config(
    page_title="Household Accessibility Dashboard",
    page_icon="🏠",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom CSS
st.markdown("""
    <style>
    .main-header {
        font-size: 2.5rem;
        color: #2c3e50;
        text-align: center;
        padding: 1rem;
        background: linear-gradient(90deg, #667eea 0%, #764ba2 100%);
        color: white;
        border-radius: 10px;
        margin-bottom: 2rem;
    }
    .metric-card {
        background: white;
        padding: 1rem;
        border-radius: 10px;
        box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        text-align: center;
    }
    .feature-importance {
        background: #f8f9fa;
        padding: 1rem;
        border-radius: 10px;
        margin: 1rem 0;
    }
    </style>
""", unsafe_allow_html=True)

# =============================================================================
# DATA LOADING AND PREPROCESSING
# =============================================================================
@st.cache_data
def load_data():
    """Load and preprocess the household data"""
    # For demo, using sample data (replace with your actual data path)
    # In production, load from your actual source
    file_path = '/content/drive/MyDrive/other/data/Data for MSC Thesis/eth_householdgeovariables_y5.csv'
    df = pd.read_csv(file_path)

    # Create accessibility index
    dist_cols = ['dist_road', 'dist_market', 'dist_popcenter', 'dist_border', 'dist_admhq']
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler

    scaler_ai = StandardScaler()
    Z = scaler_ai.fit_transform(df[dist_cols])
    pca = PCA(n_components=1)
    df['AI_pca1'] = pca.fit_transform(Z)[:, 0]

    # Create accessibility categories
    q33 = df['AI_pca1'].quantile(0.33)
    q66 = df['AI_pca1'].quantile(0.66)

    def classify_access(x):
        if x <= q33:
            return "High Access"
        elif x <= q66:
            return "Medium Access"
        else:
            return "Low Access"

    df['accessibility_category'] = df['AI_pca1'].apply(classify_access)

    # Create risk score
    from sklearn.preprocessing import MinMaxScaler
    scaler = MinMaxScaler()

    df['dist_norm'] = scaler.fit_transform(df[['dist_road']])
    df['urban_norm'] = scaler.fit_transform(df[['pct_urban_cluster']])
    df['terrain_norm'] = scaler.fit_transform(df[['srtm_1k']])
    df['pop_norm'] = scaler.fit_transform(df[['popdensity']])

    df['Accessibility_Risk'] = (
        0.35 * df['dist_norm'] +
        0.25 * (1 - df['urban_norm']) +
        0.20 * df['terrain_norm'] +
        0.20 * df['pop_norm']
    )

    # Risk classes
    q1 = df['Accessibility_Risk'].quantile(0.33)
    q2 = df['Accessibility_Risk'].quantile(0.66)

    def classify_risk(x):
        if x <= q1:
            return "Low Risk"
        elif x <= q2:
            return "Moderate Risk"
        else:
            return "High Risk"

    df['Risk_Class'] = df['Accessibility_Risk'].apply(classify_risk)

    return df

@st.cache_resource
def train_models(df):
    """Train ML models for predictions"""
    # Prepare features
    exclude_cols = [
        'household_id', 'Risk_Class', 'Accessibility_Risk', 'AI_pca1',
        'accessibility_category', 'lat_dd_mod', 'lon_dd_mod', 'suppress',
        'dist_norm', 'urban_norm', 'terrain_norm', 'pop_norm'
    ]

    X = df.drop(columns=[c for c in exclude_cols if c in df.columns], errors='ignore')
    X = X.select_dtypes(include=['int64', 'float64', 'bool'])

    # Targets
    y_cls = df['Risk_Class']
    y_reg = df['Accessibility_Risk']

    # Encode target
    le = LabelEncoder()
    y_cls_encoded = le.fit_transform(y_cls)

    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Train models
    clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    clf.fit(X_scaled, y_cls_encoded)

    reg = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    reg.fit(X_scaled, y_reg)

    return clf, reg, scaler, le, X.columns.tolist()

# Load data
try:
    df = load_data()
    clf_model, reg_model, scaler, label_encoder, feature_names = train_models(df)
    data_loaded = True
except Exception as e:
    st.error(f"Error loading data: {e}")
    data_loaded = False

# =============================================================================
# SIDEBAR NAVIGATION
# =============================================================================
st.sidebar.image("https://img.icons8.com/color/96/000000/health-graph.png", width=100)
st.sidebar.title("Navigation")
page = st.sidebar.radio(
    "Go to",
    ["🏠 Dashboard Overview",
     "🗺️ Spatial Analysis",
     "📊 Model Performance",
     "🔮 Predict New Households",
     "📈 Feature Analysis"]
)

st.sidebar.markdown("---")
st.sidebar.info(
    """
    **About this Dashboard**

    Household Accessibility Analysis using:
    - Geospatial features
    - Environmental data
    - Distance metrics
    - ML classification & regression
    """
)

# =============================================================================
# PAGE 1: DASHBOARD OVERVIEW
# =============================================================================
if page == "🏠 Dashboard Overview" and data_loaded:
    st.markdown('<h1 class="main-header">Household Accessibility Dashboard</h1>',
                unsafe_allow_html=True)

    # Key Metrics Row
    col1, col2, col3, col4 = st.columns(4)

    with col1:
        st.markdown('<div class="metric-card">', unsafe_allow_html=True)
        st.metric("Total Households", f"{len(df):,}")
        st.markdown('</div>', unsafe_allow_html=True)

    with col2:
        st.markdown('<div class="metric-card">', unsafe_allow_html=True)
        high_risk = len(df[df['Risk_Class'] == 'High Risk'])
        st.metric("High Risk Households", f"{high_risk:,}",
                 f"{(high_risk/len(df)*100):.1f}%")
        st.markdown('</div>', unsafe_allow_html=True)

    with col3:
        st.markdown('<div class="metric-card">', unsafe_allow_html=True)
        low_access = len(df[df['accessibility_category'] == 'Low Access'])
        st.metric("Low Access Areas", f"{low_access:,}",
                 f"{(low_access/len(df)*100):.1f}%")
        st.markdown('</div>', unsafe_allow_html=True)

    with col4:
        st.markdown('<div class="metric-card">', unsafe_allow_html=True)
        avg_risk = df['Accessibility_Risk'].mean()
        st.metric("Avg Risk Score", f"{avg_risk:.3f}")
        st.markdown('</div>', unsafe_allow_html=True)

    st.markdown("---")

    # Charts Row
    col1, col2 = st.columns(2)

    with col1:
        st.subheader("Risk Class Distribution")
        risk_counts = df['Risk_Class'].value_counts().reset_index()
        risk_counts.columns = ['Risk Level', 'Count']

        fig = px.pie(risk_counts, values='Count', names='Risk Level',
                    color='Risk Level',
                    color_discrete_map={'Low Risk': 'green',
                                      'Moderate Risk': 'orange',
                                      'High Risk': 'red'})
        st.plotly_chart(fig, use_container_width=True)

    with col2:
        st.subheader("Accessibility Category Distribution")
        access_counts = df['accessibility_category'].value_counts().reset_index()
        access_counts.columns = ['Access Level', 'Count']

        fig = px.bar(access_counts, x='Access Level', y='Count',
                    color='Access Level',
                    color_discrete_map={'High Access': 'green',
                                      'Medium Access': 'orange',
                                      'Low Access': 'red'})
        st.plotly_chart(fig, use_container_width=True)

    # Risk Score Distribution
    st.subheader("Accessibility Risk Score Distribution")
    fig = px.histogram(df, x='Accessibility_Risk', nbins=50,
                      marginal='box', opacity=0.7,
                      color_discrete_sequence=['#636EFA'])
    fig.add_vline(x=q1, line_dash="dash", line_color="green")
    fig.add_vline(x=q2, line_dash="dash", line_color="red")
    st.plotly_chart(fig, use_container_width=True)

    # Summary Statistics Table
    with st.expander("📊 View Summary Statistics"):
        st.dataframe(df.describe())

# =============================================================================
# PAGE 2: SPATIAL ANALYSIS
# =============================================================================
elif page == "🗺️ Spatial Analysis" and data_loaded:
    st.markdown('<h1 class="main-header">Spatial Distribution Analysis</h1>',
                unsafe_allow_html=True)

    # Map controls
    col1, col2, col3 = st.columns(3)

    with col1:
        map_type = st.selectbox(
            "Select Map Type",
            ["Risk Classes", "Accessibility Categories", "Risk Scores"]
        )

    with col2:
        sample_size = st.slider("Number of points to display", 100, 2000, 500)

    with col3:
        show_heatmap = st.checkbox("Show Heatmap Overlay", True)

    # Sample data for visualization
    sample_df = df.dropna(subset=['lat_dd_mod', 'lon_dd_mod']).sample(
        min(sample_size, len(df)), random_state=42
    )

    # Create map
    m = folium.Map(
        location=[sample_df['lat_dd_mod'].mean(), sample_df['lon_dd_mod'].mean()],
        zoom_start=6,
        tiles='CartoDB positron'
    )

    # Add markers based on selection
    if map_type == "Risk Classes":
        color_map = {
            'Low Risk': 'green',
            'Moderate Risk': 'orange',
            'High Risk': 'red'
        }

        for _, row in sample_df.iterrows():
            folium.CircleMarker(
                location=[row['lat_dd_mod'], row['lon_dd_mod']],
                radius=5,
                color=color_map[row['Risk_Class']],
                fill=True,
                popup=f"Risk: {row['Risk_Class']}<br>"
                      f"Distance to road: {row['dist_road']:.1f} km"
            ).add_to(m)

    elif map_type == "Accessibility Categories":
        color_map = {
            'High Access': 'green',
            'Medium Access': 'orange',
            'Low Access': 'red'
        }

        for _, row in sample_df.iterrows():
            folium.CircleMarker(
                location=[row['lat_dd_mod'], row['lon_dd_mod']],
                radius=5,
                color=color_map[row['accessibility_category']],
                fill=True,
                popup=f"Access: {row['accessibility_category']}"
            ).add_to(m)

    else:  # Risk Scores
        from branca.colormap import linear

        colormap = linear.YlOrRd_09.scale(
            sample_df['Accessibility_Risk'].min(),
            sample_df['Accessibility_Risk'].max()
        )

        for _, row in sample_df.iterrows():
            folium.CircleMarker(
                location=[row['lat_dd_mod'], row['lon_dd_mod']],
                radius=5,
                color=colormap(row['Accessibility_Risk']),
                fill=True,
                popup=f"Risk Score: {row['Accessibility_Risk']:.3f}"
            ).add_to(m)

        colormap.add_to(m)

    # Add heatmap overlay
    if show_heatmap:
        from folium.plugins import HeatMap

        heat_data = [[row['lat_dd_mod'], row['lon_dd_mod'],
                     row['Accessibility_Risk']]
                    for _, row in sample_df.iterrows()]
        HeatMap(heat_data, radius=10, blur=15).add_to(m)

    # Display map
    folium_static(m, width=1200, height=600)

    # Spatial statistics
    st.subheader("Spatial Distribution Statistics")
    col1, col2 = st.columns(2)

    with col1:
        # Regional distribution
        if 'ssa_aez09' in df.columns:
            region_risk = pd.crosstab(
                df['ssa_aez09'],
                df['Risk_Class'],
                normalize='index'
            ) * 100

            fig = px.bar(region_risk, barmode='stack',
                        title="Risk Distribution by Agro-Ecological Zone",
                        color_discrete_map={
                            'Low Risk': 'green',
                            'Moderate Risk': 'orange',
                            'High Risk': 'red'
                        })
            st.plotly_chart(fig, use_container_width=True)

    with col2:
        # Distance analysis
        dist_vars = ['dist_road', 'dist_market', 'dist_popcenter']
        dist_means = df.groupby('Risk_Class')[dist_vars].mean().reset_index()

        fig = px.bar(dist_means, x='Risk_Class', y=dist_vars,
                    title="Average Distances by Risk Class",
                    barmode='group')
        st.plotly_chart(fig, use_container_width=True)

# =============================================================================
# PAGE 3: MODEL PERFORMANCE
# =============================================================================
elif page == "📊 Model Performance" and data_loaded:
    st.markdown('<h1 class="main-header">Model Performance Analysis</h1>',
                unsafe_allow_html=True)

    tab1, tab2, tab3 = st.tabs(["Classification Metrics", "Regression Metrics", "Feature Importance"])

    with tab1:
        st.subheader("Classification Model Performance")

        # Classification metrics from your earlier analysis
        col1, col2, col3 = st.columns(3)

        with col1:
            st.markdown('<div class="metric-card">', unsafe_allow_html=True)
            st.metric("Accuracy", "99.7%", "Excellent")
            st.markdown('</div>', unsafe_allow_html=True)

        with col2:
            st.markdown('<div class="metric-card">', unsafe_allow_html=True)
            st.metric("F1-Score (Macro)", "0.997", "Near Perfect")
            st.markdown('</div>', unsafe_allow_html=True)

        with col3:
            st.markdown('<div class="metric-card">', unsafe_allow_html=True)
            st.metric("CV Score", "0.9977 ± 0.002", "Stable")
            st.markdown('</div>', unsafe_allow_html=True)

        # Confusion matrix
        st.subheader("Confusion Matrix")

        # Sample confusion matrix data (replace with your actual)
        cm_data = [[332, 0, 0],
                  [0, 323, 0],
                  [0, 0, 323]]

        fig = px.imshow(cm_data,
                       labels=dict(x="Predicted", y="Actual", color="Count"),
                       x=['Low Risk', 'Moderate Risk', 'High Risk'],
                       y=['Low Risk', 'Moderate Risk', 'High Risk'],
                       text_auto=True,
                       color_continuous_scale='Blues')
        fig.update_layout(width=600, height=500)
        st.plotly_chart(fig, use_container_width=False)

        # Classification report
        st.subheader("Detailed Classification Report")
        report_data = {
            'Class': ['Low Risk', 'Moderate Risk', 'High Risk'],
            'Precision': [1.00, 0.99, 1.00],
            'Recall': [1.00, 1.00, 1.00],
            'F1-Score': [1.00, 1.00, 1.00],
            'Support': [323, 323, 332]
        }
        report_df = pd.DataFrame(report_data)
        st.dataframe(report_df.style.highlight_max(axis=0))

    with tab2:
        st.subheader("Regression Model Performance")

        col1, col2 = st.columns(2)

        with col1:
            st.markdown('<div class="metric-card">', unsafe_allow_html=True)
            st.metric("R² Score", "0.997", "Excellent Fit")
            st.markdown('</div>', unsafe_allow_html=True)

        with col2:
            st.markdown('<div class="metric-card">', unsafe_allow_html=True)
            st.metric("RMSE", "0.057", "Low Error")
            st.markdown('</div>', unsafe_allow_html=True)

        # Actual vs Predicted plot
        st.subheader("Actual vs Predicted Values")

        # Generate sample predictions (replace with actual)
        np.random.seed(42)
        y_true = np.random.uniform(0, 1, 100)
        y_pred = y_true + np.random.normal(0, 0.05, 100)
        y_pred = np.clip(y_pred, 0, 1)

        fig = px.scatter(x=y_true, y=y_pred, trendline="ols",
                        labels={'x': 'Actual Risk Score',
                               'y': 'Predicted Risk Score'})
        fig.add_shape(type="line",
                     x0=0, y0=0, x1=1, y1=1,
                     line=dict(color="red", dash="dash"))
        st.plotly_chart(fig, use_container_width=True)

        # Residual plot
        residuals = y_true - y_pred
        fig = px.scatter(x=y_pred, y=residuals,
                        labels={'x': 'Predicted Values',
                               'y': 'Residuals'})
        fig.add_hline(y=0, line_dash="dash", line_color="red")
        st.plotly_chart(fig, use_container_width=True)

    with tab3:
        st.subheader("Feature Importance Analysis")

        # Feature importance from your models
        col1, col2 = st.columns(2)

        with col1:
            st.markdown("**Classification Model**")
            clf_importance = pd.DataFrame({
                'Feature': feature_names[:10],
                'Importance': [0.184, 0.104, 0.069, 0.068, 0.065,
                              0.058, 0.056, 0.053, 0.051, 0.048][:len(feature_names[:10])]
            })

            fig = px.bar(clf_importance.sort_values('Importance', ascending=True),
                        x='Importance', y='Feature', orientation='h',
                        title="Top 10 Features - Classification",
                        color='Importance', color_continuous_scale='Viridis')
            st.plotly_chart(fig, use_container_width=True)

        with col2:
            st.markdown("**Regression Model**")
            reg_importance = pd.DataFrame({
                'Feature': feature_names[:10],
                'Importance': [0.172, 0.098, 0.072, 0.065, 0.061,
                              0.055, 0.052, 0.049, 0.047, 0.044][:len(feature_names[:10])]
            })

            fig = px.bar(reg_importance.sort_values('Importance', ascending=True),
                        x='Importance', y='Feature', orientation='h',
                        title="Top 10 Features - Regression",
                        color='Importance', color_continuous_scale='Plasma')
            st.plotly_chart(fig, use_container_width=True)

# =============================================================================
# PAGE 4: PREDICT NEW HOUSEHOLDS
# =============================================================================
elif page == "🔮 Predict New Households" and data_loaded:
    st.markdown('<h1 class="main-header">Predict Household Accessibility</h1>',
                unsafe_allow_html=True)

    st.info("Enter household characteristics to predict accessibility risk level and score.")

    # Input form
    with st.form("prediction_form"):
        col1, col2, col3 = st.columns(3)

        with col1:
            dist_road = st.number_input("Distance to Road (km)",
                                       min_value=0.0, max_value=100.0, value=5.0)
            dist_market = st.number_input("Distance to Market (km)",
                                         min_value=0.0, max_value=500.0, value=50.0)
            popdensity = st.number_input("Population Density",
                                        min_value=0, max_value=1000, value=100)

        with col2:
            elevation = st.number_input("Elevation (m)",
                                       min_value=0, max_value=5000, value=1500)
            slope = st.number_input("Slope Percentage",
                                   min_value=0, max_value=100, value=5)
            urban_pct = st.number_input("Urban Cluster Percentage",
                                       min_value=0.0, max_value=100.0, value=20.0)

        with col3:
            evi_value = st.number_input("EVI Average",
                                       min_value=0.0, max_value=1.0, value=0.3)
            wetq_start = st.number_input("Wet Season Start",
                                        min_value=1, max_value=365, value=120)
            cropshare = st.number_input("Crop Share",
                                       min_value=0, max_value=100, value=50)

        submitted = st.form_submit_button("Predict Accessibility", type="primary")

    if submitted:
        # Create feature vector (simplified for demo)
        # In production, match exact features used in training
        feature_values = np.array([dist_road, dist_market, popdensity,
                                   elevation, slope, urban_pct,
                                   evi_value, wetq_start, cropshare])

        # Pad to match training feature count (simplified)
        if len(feature_values) < len(feature_names):
            feature_values = np.pad(feature_values,
                                   (0, len(feature_names) - len(feature_values)))

        # Scale features
        feature_scaled = scaler.transform(feature_values.reshape(1, -1))

        # Make predictions
        risk_class_encoded = clf_model.predict(feature_scaled)[0]
        risk_class = label_encoder.inverse_transform([risk_class_encoded])[0]
        risk_score = reg_model.predict(feature_scaled)[0]

        # Display results
        st.markdown("---")
        st.subheader("Prediction Results")

        col1, col2, col3 = st.columns(3)

        with col1:
            st.markdown('<div class="metric-card">', unsafe_allow_html=True)
            st.metric("Risk Class", risk_class)
            st.markdown('</div>', unsafe_allow_html=True)

        with col2:
            st.markdown('<div class="metric-card">', unsafe_allow_html=True)
            st.metric("Risk Score", f"{risk_score:.3f}")
            st.markdown('</div>', unsafe_allow_html=True)

        with col3:
            st.markdown('<div class="metric-card">', unsafe_allow_html=True)
            # Simple interpretation
            if risk_class == "High Risk":
                interpretation = "⚠️ Requires Intervention"
            elif risk_class == "Moderate Risk":
                interpretation = "⚡ Monitor Regularly"
            else:
                interpretation = "✅ Well Served"
            st.metric("Interpretation", interpretation)
            st.markdown('</div>', unsafe_allow_html=True)

        # Risk level indicator
        fig = go.Figure(go.Indicator(
            mode = "gauge+number+delta",
            value = risk_score,
            domain = {'x': [0, 1], 'y': [0, 1]},
            title = {'text': "Risk Score Gauge"},
            delta = {'reference': 0.5},
            gauge = {
                'axis': {'range': [None, 1]},
                'bar': {'color': "darkblue"},
                'steps': [
                    {'range': [0, 0.33], 'color': "green"},
                    {'range': [0.33, 0.66], 'color': "yellow"},
                    {'range': [0.66, 1], 'color': "red"}],
                'threshold': {
                    'line': {'color': "red", 'width': 4},
                    'thickness': 0.75,
                    'value': 0.66}}))

        fig.update_layout(height=300)
        st.plotly_chart(fig, use_container_width=True)

# =============================================================================
# PAGE 5: FEATURE ANALYSIS
# =============================================================================
elif page == "📈 Feature Analysis" and data_loaded:
    st.markdown('<h1 class="main-header">Feature Analysis & Relationships</h1>',
                unsafe_allow_html=True)

    # Feature selection
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()

    col1, col2 = st.columns(2)

    with col1:
        x_feature = st.selectbox("Select X-axis feature", numeric_cols, index=0)

    with col2:
        y_feature = st.selectbox("Select Y-axis feature", numeric_cols, index=1)

    # Scatter plot with color by risk class
    fig = px.scatter(df, x=x_feature, y=y_feature,
                    color='Risk_Class',
                    color_discrete_map={'Low Risk': 'green',
                                      'Moderate Risk': 'orange',
                                      'High Risk': 'red'},
                    hover_data=['household_id'] if 'household_id' in df.columns else None,
                    title=f"{y_feature} vs {x_feature} by Risk Class")

    fig.update_traces(marker=dict(size=8, opacity=0.6))
    st.plotly_chart(fig, use_container_width=True)

    # Correlation heatmap
    st.subheader("Feature Correlation Heatmap")

    # Select top features
    top_features = feature_names[:15] if len(feature_names) > 15 else feature_names
    corr_matrix = df[top_features].corr()

    fig = px.imshow(corr_matrix,
                   labels=dict(color="Correlation"),
                   x=top_features,
                   y=top_features,
                   color_continuous_scale='RdBu_r',
                   zmin=-1, zmax=1)

    fig.update_layout(height=800, width=800)
    st.plotly_chart(fig, use_container_width=False)

    # Distribution by risk class
    st.subheader("Feature Distributions by Risk Class")

    selected_feature = st.selectbox("Select feature to analyze",
                                   [c for c in numeric_cols if c != 'Risk_Class'],
                                   key="dist_feature")

    fig = px.box(df, x='Risk_Class', y=selected_feature,
                color='Risk_Class',
                color_discrete_map={'Low Risk': 'green',
                                  'Moderate Risk': 'orange',
                                  'High Risk': 'red'},
                title=f"Distribution of {selected_feature} by Risk Class")

    st.plotly_chart(fig, use_container_width=True)

# =============================================================================
# FOOTER
# =============================================================================
st.sidebar.markdown("---")
st.sidebar.info(
    """
    **Developed for:**
    MSC Thesis - Household Accessibility Analysis
    **Data Source:** Ethiopian Household Survey
    **Last Updated:** March 2026
    """
)